### Imports

In [1]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
from torchvision.transforms import transforms

import matplotlib.pyplot as plt
from pathlib import Path

### Load ResNet18 Model + GRU

In [2]:
class Experiment3(nn.Module):
    def __init__(self, num_classes=4):
        super(Experiment3, self).__init__()
        # load pretrained model
        resnet_model = resnet18(weights=ResNet18_Weights.DEFAULT)

        # remove last fully connected layer
        self.feature_extractor = nn.Sequential(*list(resnet_model.children())[:-1])

        # define GRU
        self.gru = nn.GRU(input_size=512, hidden_size=128, batch_first=True) # hidden_size möglicherweise anpassen

        # final classifier
        self.fc = nn.Linear(128, num_classes) 


    def forward(self,x):
        batch_size, seq_len, channels, height, width = x.size()

        # use ResNet for each picture in the sequence
        x = x.view(batch_size * seq_len, channels, height, width)
        with torch.no_grad(): # ResNet eingefroren
            features = self.feature_extractor(x)

        # bring features into right sequential form
        features = torch.flatten(features, 1)
        features = features.view(batch_size, seq_len, -1)

        gru_out, hn = self.gru(features)

        last_hidden_state = hn[-1]

        output = self.fc(last_hidden_state)
        return output

### Load Dataset

In [5]:
# define path 
DATASET_PATH = "/Users/gwen/Desktop/video_frames_data"
CSV_PATH = os.path.join(DATASET_PATH, "metadata.csv")

# load dataset
class VideoDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.df = dataframe 
        self.root_dir = root_dir
        self.transform = transform
        self.seq_len = 8 

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        video_folder = os.path.join(self.root_dir, row['frame_dir'] )

        video_class = int(row['class_id']) 

        img_sequence = sorted(os.listdir(video_folder))


        frames = []
        for img_name in img_sequence:
            img_path = os.path.join(video_folder, img_name)
            image = Image.open(img_path).convert('RGB') # to make 100% sure
            if self.transform:
                image = self.transform(image)
            frames.append(image)

        return torch.stack(frames), video_class


# define transformations for our dataset
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

# load full csv
full_df = pd.read_csv(CSV_PATH)

full_df['frame_dir'] = full_df['frame_dir'].str.replace('\\', '/', regex=False) # wegen Windows und Mac

# load training data
train_df = full_df[full_df['split'] == 'train'].reset_index(drop=True)
# load validation data
val_df = full_df[full_df['split'] == 'val'].reset_index(drop=True)

# Dataset and Loader
train_dataset = VideoDataset(dataframe=train_df, root_dir=DATASET_PATH, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataset = VideoDataset(dataframe=val_df, root_dir=DATASET_PATH, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


### Training Loop

In [4]:
# training loop 
def train(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    # determine device
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

    for epoch in range(num_epochs):
        # training mode
        model.train()

        running_loss = 0.0
        running_corrects = 0

        # iterate training data loader
        for inputs, labels in train_loader:
            # put to device
            inputs = inputs.to(device)
            labels = labels.to(device)

            # reset gradients 
            optimizer.zero_grad()

            # forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            # backward pass
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # epoch training loss and accuracy 
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = running_corrects.float() / len(train_loader.dataset)


        ## Evaluation
        model.eval()
        running_loss = 0.0
        running_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # forward pass
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        # validation loss and accuracy 
        val_loss = running_loss / len(val_loader.dataset)
        val_acc = running_corrects.float() / len(val_loader.dataset)

        print(f'Epoch [{epoch+1}/{num_epochs}], train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, val loss: {val_loss:.4f}, val acc: {val_acc:.4f} \n')

In [5]:
# train model
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
model = Experiment3(num_classes=4).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001) # SGD durch Adam ersetzt, da es anscheinend für GRUs stabiler und schneller sein soll

train(model, train_loader, val_loader, criterion, optimizer, num_epochs=10)

# save checkpoint of model
checkpoint = {
    "Experiment3_state_dict": model.state_dict(),
    "optimizer_Experiment3_state_dict": optimizer.state_dict(),
    "epoch": 10,
}

torch.save(
    checkpoint,
    "checkpoints/Experiment3_checkpoint.pth"
)

Epoch [1/10], train loss: 1.0466, train acc: 0.4667, val loss: 0.9362, val acc: 0.5018 

Epoch [2/10], train loss: 0.8316, train acc: 0.5660, val loss: 0.8419, val acc: 0.5474 

Epoch [3/10], train loss: 0.8188, train acc: 0.5742, val loss: 0.8206, val acc: 0.5947 

Epoch [4/10], train loss: 0.7785, train acc: 0.6137, val loss: 0.7663, val acc: 0.5702 

Epoch [5/10], train loss: 0.7823, train acc: 0.6197, val loss: 0.7765, val acc: 0.5895 

Epoch [6/10], train loss: 0.7382, train acc: 0.6419, val loss: 0.7925, val acc: 0.5789 

Epoch [7/10], train loss: 0.7142, train acc: 0.6648, val loss: 0.7624, val acc: 0.6123 

Epoch [8/10], train loss: 0.6904, train acc: 0.6716, val loss: 0.7732, val acc: 0.6070 

Epoch [9/10], train loss: 0.7044, train acc: 0.6667, val loss: 0.7620, val acc: 0.5982 

Epoch [10/10], train loss: 0.6827, train acc: 0.6847, val loss: 0.7735, val acc: 0.6123 



RuntimeError: Parent directory checkpoints does not exist.